# AC-Zero — Scheduler Runner (Kaggle)

The **single** notebook the GitHub Actions scheduler launches for every task. It
reads a secret-free `runtime_config.json` the scheduler dropped next to it,
branches on `mode` (`generation` / `annotation` / `training`), reports
heartbeats/status to the private Hugging Face state repo, and stops cleanly at a
soft runtime deadline (well before Kaggle's ~12 h hard kill).

**Do not edit the config cells by hand** — the scheduler owns them. Manual
control happens by editing the HF `queue.yaml` or via the workflow dispatch.

The Hugging Face token arrives through the private Kaggle **runtime-secrets**
dataset (`/kaggle/input/runtime-secrets/hf_token.txt`); it is never printed,
saved to `/kaggle/working`, or written into any output.

## 1. Install AC-Zero from GitHub

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/HkHk2Prod/AlphaAC.git"
REPO_BRANCH = "main"

def pip_install(*args):
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off", *args],
        capture_output=True, text=True,
    )
    if proc.returncode:
        print(proc.stdout, proc.stderr, sep="\n"); proc.check_returncode()

pip_install("--no-deps", f"git+{REPO_URL}@{REPO_BRANCH}")
pip_install("numpy>=1.26", "pydantic>=2", "pyyaml>=6", "typer>=0.12", "rich>=13",
            "gymnasium>=0.29", "matplotlib>=3.8", "huggingface_hub>=1.21")
print("Installed ac_zero from", REPO_BRANCH)

## 2. Runtime config, Hugging Face login, run reporter

In [ ]:
import os

from ac_zero.scheduler.notebook import RunReporter, load_runtime_config, login_from_secret_dataset

CFG = load_runtime_config("runtime_config.json")
TASK_ID = CFG["task_id"]; RUN_ID = CFG["run_id"]; MODE = CFG["mode"]
MAX_RUNTIME_MIN = int(CFG.get("max_runtime_minutes", 705))
STATE_REPO = CFG["hf_state_repo_id"]; STATE_REPO_TYPE = CFG.get("hf_state_repo_type", "dataset")
OPTS = CFG.get("config", {})
print(f"task={TASK_ID} run={RUN_ID} mode={MODE} max_runtime_min={MAX_RUNTIME_MIN}")

# Read the HF token from the private Kaggle dataset and log in (never printed).
_hf_token = login_from_secret_dataset()
os.environ["HF_TOKEN"] = _hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token

reporter = RunReporter(STATE_REPO, run_id=RUN_ID, task_id=TASK_ID,
                       token=_hf_token, repo_type=STATE_REPO_TYPE)
reporter.started(extra={"accelerator": CFG.get("accelerator")})
print("reported run start to", STATE_REPO)

## 3. Soft deadline + heartbeat / stop-signal thread

In [ ]:
import threading
import time

START = time.monotonic()
SAFETY_MARGIN_MIN = 10
DEADLINE = START + max(1, MAX_RUNTIME_MIN - SAFETY_MARGIN_MIN) * 60
HEARTBEAT_EVERY_S = 300  # 5 min

# How long a job may run before the watchdog terminates it. Training reserves
# extra head-room on top of that: after its last iteration it still writes plots
# and the certificate and pushes the checkpoint bundle to Hugging Face, so it must
# finish the loop early enough to do so rather than be SIGTERM'd mid-upload.
JOB_BUDGET_MIN = max(1, MAX_RUNTIME_MIN - SAFETY_MARGIN_MIN)
TRAIN_FLUSH_MARGIN_MIN = 10
TRAIN_BUDGET_MIN = max(1, JOB_BUDGET_MIN - TRAIN_FLUSH_MARGIN_MIN)

stop_event = threading.Event()          # set on deadline OR operator stop request
deadline_hit = threading.Event()

def _watchdog():
    while not stop_event.is_set():
        if time.monotonic() >= DEADLINE:
            print("[watchdog] soft deadline reached; requesting stop.")
            deadline_hit.set(); stop_event.set(); break
        try:
            reporter.heartbeat(extra={"elapsed_min": round((time.monotonic()-START)/60, 1)})
            if reporter.should_stop():
                print("[watchdog] operator stop requested via queue; requesting stop.")
                stop_event.set(); break
        except Exception as exc:  # never let a transient HF blip kill the run
            print("[watchdog] heartbeat/stop-check skipped:", exc)
        stop_event.wait(HEARTBEAT_EVERY_S)

_hb = threading.Thread(target=_watchdog, daemon=True); _hb.start()
print(f"watchdog running: soft deadline in {MAX_RUNTIME_MIN - SAFETY_MARGIN_MIN} min, "
      f"heartbeat every {HEARTBEAT_EVERY_S//60} min.")

## 4. Prepare inputs and build the mode command

The same notebook serves every task; `mode` and `config` from the runtime config
select what runs. Each mode maps to an `aczero` CLI subcommand. Annotation, solve
and training pull the grown dataset (and, for the `navigation` reward, its
`strict-ac` annotations) from the shared HF bucket first; training materializes
`config` into a YAML the CLI reads, so no repo `configs/` files are needed on
Kaggle. Generation resumes the bucket dataset and **writes it back**; annotation
writes its refined annotations back; training warm-starts from the best model
under `model_checkpoints/<name>/` and pushes its own bundle back — so each
scheduled run continues the last.

In [ ]:
from pathlib import Path

import yaml

HF_BUCKET_DEFAULT = "HkHk2Prod/alphaac-data"
DATA_DIR = Path("data/generated"); DATA_DIR.mkdir(parents=True, exist_ok=True)

def _opt(name, default=None):
    return OPTS.get(name, default)

def _bucket():
    return (OPTS.get("dataset") or {}).get("bucket") or HF_BUCKET_DEFAULT

def pull(remote_name, local_path):
    from ac_zero.datasets.hub import download_dataset
    return download_dataset(local_path, remote_name=remote_name, bucket=_bucket(), missing_ok=True)

# (local_path, remote_name) pairs pushed back to the bucket after a successful run.
UPLOADS = []

def push_outputs():
    # Upload each (local, remote) in UPLOADS; return the remote names pushed.
    from ac_zero.datasets.hub import upload_dataset
    pushed = []
    for local_path, remote_name in UPLOADS:
        p = Path(local_path)
        if not p.exists():
            print(f"[write-back] skip {remote_name}: {p} not found"); continue
        upload_dataset(p, remote_name=remote_name, bucket=_bucket())
        print(f"[write-back] pushed {p.name} -> {_bucket()}/{remote_name}")
        pushed.append(remote_name)
    return pushed

def prepare_and_build(mode):
    rank = int(_opt("rank", 2))
    moveset = str(_opt("moveset", "strict-ac"))
    # Two datasets can live in the bucket. `ball_rank{rank}` is grown closest-first, so
    # every distance in it is a proven optimum and its shells are complete; `train_rank
    # {rank}` is the older length-first grow, whose distances are searched for afterwards
    # and are only upper bounds. Training reads the ball; a task can name the other with
    # `dataset.stem: train_rank2`.
    stem = str((OPTS.get("dataset") or {}).get("stem") or f"ball_rank{rank}")
    ds_name = f"{stem}.groups.json"
    ann_name = f"{stem}.{moveset}.annotations.json"
    grown_name = f"train_rank{rank}.groups.json"
    grown_ann = f"train_rank{rank}.{moveset}.annotations.json"

    if mode == "ball":
        # Closest-first generation: walk outward from the trivial group by the inverses
        # of the move set's moves, breadth-first. Resume needs both files -- the groups
        # are the queue (expanded in discovery order) and the annotations are the
        # distances they were discovered at -- and both are pushed back.
        output = str(_opt("output", f"/kaggle/working/{ds_name}"))
        ann_output = str(Path(output).parent / ann_name)
        pull(ds_name, output)
        pull(ann_name, ann_output)
        UPLOADS.append((output, ds_name))
        UPLOADS.append((ann_output, ann_name))
        return ["aczero", "dataset", "ball",
                "--rank", str(rank),
                "--moveset", moveset,
                "--target", str(_opt("target", 100_000_000)),
                # Fill the run: expand until the soft deadline, not just the target.
                "--minutes", str(JOB_BUDGET_MIN),
                "--checkpoint-every", str(_opt("checkpoint_every", 50_000)),
                "--workers", str(_opt("workers", 0)),
                "--output", output]

    if mode == "generation":
        # The older length-first grow, over the full universal move set.
        output = str(_opt("output", f"/kaggle/working/{grown_name}"))
        pull(grown_name, output)
        UPLOADS.append((output, grown_name))
        return ["aczero", "dataset", "grow",
                "--rank", str(rank),
                "--target", str(_opt("target", 100_000_000)),
                # Fill the run: grow until the soft deadline, not just the target.
                "--minutes", str(JOB_BUDGET_MIN),
                "--total-length-cap", str(_opt("total_length_cap", 0)),
                "--select", str(_opt("select", "smallest")),
                "--seed", str(_opt("seed", 0)),
                "--checkpoint-every", str(_opt("checkpoint_every", 1000)),
                "--workers", str(_opt("workers", 0)),
                "--output", output]

    if mode == "annotation":
        # Only a grown dataset needs annotating; a ball writes its distances itself.
        ds_local = DATA_DIR / grown_name
        if pull(grown_name, ds_local) is None:
            raise RuntimeError(f"annotation needs {grown_name} in bucket {_bucket()}; run generation first.")
        ann_local = DATA_DIR / grown_ann
        pull(grown_ann, ann_local)  # warm start from any existing annotations
        UPLOADS.append((ann_local, grown_ann))  # push refined annotations back
        return ["aczero", "dataset", "annotate",
                "--input", str(ds_local),
                "--moveset", moveset,
                "--max-depth", str(_opt("max_depth", 0)),
                "--workers", str(_opt("workers", 0))]

    if mode == "training":
        ds_local, ann_local = DATA_DIR / ds_name, DATA_DIR / ann_name
        if pull(ds_name, ds_local) is None:
            raise RuntimeError(f"training needs {ds_name} in bucket {_bucket()}; run ball generation first.")
        have_ann = pull(ann_name, ann_local) is not None
        reward = (OPTS.get("training") or {}).get("reward_mode")
        if reward in ("navigation", "potential") and not have_ann:
            raise RuntimeError(f"reward_mode {reward!r} needs {ann_name} in bucket; generate the ball first.")
        cfg = {k: v for k, v in OPTS.items() if k != "seed"}
        ds_cfg = {k: v for k, v in (cfg.get("dataset") or {}).items() if k != "stem"}
        ds_cfg["path"] = str(ds_local)
        if have_ann:
            ds_cfg["annotations"] = str(ann_local)
        cfg["dataset"] = ds_cfg
        cfg_path = Path("/kaggle/working/_train_config.yaml")
        cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
        return ["aczero", "train",
                "--config", str(cfg_path),
                "--seed", str(_opt("seed", 0)),
                # Fill the run: the wall-clock budget, not `training.iterations`,
                # ends it -- at an iteration boundary, with time left to upload.
                "--minutes", str(TRAIN_BUDGET_MIN),
                # Continue this checkpoint lineage on HF: warm-start from the best
                # model already there, then push this run's best model + plots back.
                "--download-checkpoint",
                "--upload-checkpoints",
                "--checkpoint-bucket", _bucket()]

    if mode == "solve":
        ds_local = DATA_DIR / ds_name
        if pull(ds_name, ds_local) is None:
            raise RuntimeError(f"solve needs {ds_name} in bucket {_bucket()}; run ball generation first.")
        return ["aczero", "solve", "--presentation", str(ds_local), "--agent", str(_opt("agent", "greedy"))]

    raise ValueError(f"unknown mode: {mode!r}")

COMMAND = prepare_and_build(MODE)
print("running:", " ".join(COMMAND))


## 5. Run the job under the deadline, then report status

In [ ]:

status = "finished"; error = None
proc = subprocess.Popen(COMMAND, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    while True:
        if proc.poll() is not None:
            break
        if stop_event.is_set():
            print("[main] stop requested — terminating job so it can checkpoint & exit.")
            proc.terminate()
            try:
                proc.wait(timeout=120)
            except subprocess.TimeoutExpired:
                proc.kill()
            status = "stopped" if not deadline_hit.is_set() else "finished"
            break
        line = proc.stdout.readline() if proc.stdout else ""
        if line:
            print(line, end="")
        else:
            time.sleep(1)
    if proc.stdout:
        for line in proc.stdout:
            print(line, end="")
    rc = proc.wait()
    if rc != 0 and status == "finished":
        status = "failed"; error = f"job exited with code {rc}"
except Exception as exc:
    status = "failed"; error = repr(exc)
finally:
    stop_event.set()

# Write-back: push the grown dataset / refined annotations to the bucket so the
# next scheduled run continues from them. Only on a usable outcome (finished or a
# clean operator stop); grow/annotate checkpoint atomically, so the file is
# consistent. Training pushes its own model checkpoints via the training pipeline.
wrote_back = []
if status in ("finished", "stopped"):
    try:
        wrote_back = push_outputs()
    except Exception as exc:
        print(f"[write-back] failed: {exc}")

# A failed run still counts as a completed scheduled launch — the scheduler does
# NOT restore remaining_runs. We only report status for visibility. `wrote_back`
# records which bucket artifacts this run pushed, for observability.
reporter.finished(status=status, error=error,
                  extra={"wrote_back": wrote_back} if wrote_back else None)
print(f"run {RUN_ID} reported as {status}" + (f" ({error})" if error else ""))
if status == "failed":
    raise SystemExit(error or "job failed")